# Оптимизация памяти, типы данных Categorical, Scikit-learn Pipelines и быстрые форматы хранения.

In [53]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import os

In [54]:
olist_orders = pd.read_csv('olist_orders_dataset.csv')
olist_customers = pd.read_csv('olist_customers_dataset.csv')
olist_order_items = pd.read_csv('olist_order_items_dataset.csv')

In [55]:
df = pd.merge(left=olist_customers, right=olist_orders, how='inner', on='customer_id')
df = pd.merge(left=df, right=olist_order_items, how='inner', on='order_id')

In [56]:
to_datetime = ['order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date', 
    'shipping_limit_date']

In [57]:
for col in to_datetime:
    df[col] = pd.to_datetime(df[col], format='mixed')

### Оптимизация памяти (Categorical Data) ###

Текстовые данные о штатах и статусах заказов занимают в памяти значительно больше места, чем могли бы.

Измерение: Оценил текущее потребление памяти объединенным датасетом с помощью df.info(memory_usage='deep').

Трансформация: Перевел колонки customer_state, order_status и product_category_name в тип category.

Сравнение: Замерил объем памяти «до» и «после».

In [58]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   customer_id                    112650 non-null  str           
 1   customer_unique_id             112650 non-null  str           
 2   customer_zip_code_prefix       112650 non-null  int64         
 3   customer_city                  112650 non-null  str           
 4   customer_state                 112650 non-null  str           
 5   order_id                       112650 non-null  str           
 6   order_status                   112650 non-null  str           
 7   order_purchase_timestamp       112650 non-null  datetime64[us]
 8   order_approved_at              112635 non-null  datetime64[us]
 9   order_delivered_carrier_date   111456 non-null  datetime64[us]
 10  order_delivered_customer_date  110196 non-null  datetime64[us]
 11  order_estim

In [59]:
df[['customer_state', 'order_status']] = df[['customer_state', 'order_status']].astype('category')
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 18 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   customer_id                    112650 non-null  str           
 1   customer_unique_id             112650 non-null  str           
 2   customer_zip_code_prefix       112650 non-null  int64         
 3   customer_city                  112650 non-null  str           
 4   customer_state                 112650 non-null  category      
 5   order_id                       112650 non-null  str           
 6   order_status                   112650 non-null  category      
 7   order_purchase_timestamp       112650 non-null  datetime64[us]
 8   order_approved_at              112635 non-null  datetime64[us]
 9   order_delivered_carrier_date   111456 non-null  datetime64[us]
 10  order_delivered_customer_date  110196 non-null  datetime64[us]
 11  order_estim

Перевод customer_state и order_status в category сократил 
потребление памяти с 70.2 MB до 58.7 MB - экономия ~16%.
Колонки с ID-строками остались str: у них высокая кардинальность 
(почти уникальные значения), поэтому category там не даёт выигрыша.

### Автоматизация подготовки (Sklearn Pipelines) ###

Чтобы избежать утечки данных (data leakage) и упростить деплой, нужно собрать предобработку в один объект.

Создал ColumnTransformer, который включает:

Для численных признаков ( price, freight_value): заполнение пропусков медианой и масштабирование ( StandardScaler).

Для категориальных признаков: One-Hot Encoding или Ordinal Encoding.

Обернул это в Pipeline и протестировал его на небольшом фрагменте данных.

In [60]:
num_features = ['price', 'freight_value']
cat_features = ['customer_state', 'order_status']

In [61]:
num_transformer = Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                   ('scaler', StandardScaler())])

cat_transformer = Pipeline(steps=[('encoder', OneHotEncoder())])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features), 
    ('cat', cat_transformer, cat_features)
    ])

pipeline = Pipeline(steps=[('preprocessor', preprocessor)])

In [62]:
result = pipeline.fit_transform(df.head(100))
pd.DataFrame(result.toarray()).head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.227547,0.266920,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
1,1.743322,2.234685,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,0.365714,-0.060241,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
3,0.458134,0.385306,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
4,1.198046,0.296517,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0


Pipeline успешно обработал тестовую выборку из 100 строк:
числовые признаки (price, freight_value) стандартизированы,
категориальные (customer_state, order_status) развёрнуты в 
бинарные колонки через OneHotEncoder - итого 20 признаков на выходе.

### Высокопроизводительные запросы (pd.query) ###

В сложных системах читаемость кода и скорость фильтрации имеют решающее значение.

Реализовал сложный фильтр (заказы из штатов SP или RJ со стоимостью выше 150 BRL, сделанные в 2018 году) двумя способами:
1. Через классические булевы маски.
2. Через метод .query().

Использовал %timeit, чтобы проверить, какой метод работает быстрее.

In [63]:
%%timeit
df[(df['customer_state'].isin(['SP', 'RJ'])) & 
   (df['price'] + df['freight_value'] > 150) & 
   (df['order_purchase_timestamp'].dt.year == 2018)]

5.02 ms ± 14.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [64]:
%%timeit
df.query("customer_state in ['SP', 'RJ'] and price + freight_value > 150 and order_purchase_timestamp.dt.year == 2018")

5.89 ms ± 20.1 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Булева маска: 5.02 мс, .query(): 5.89 мс — результаты практически идентичны.
На небольших датасетах разница незначительна. .query() выигрывает 
в читаемости при сложных условиях, булевы маски - в гибкости.

### Переход на бинарные форматы (HDF5 / Parquet) ###

CSV — это текстовый формат, который теряет информацию о типах данных (datetime и category снова станут строками при чтении).

Сохранил финальный датасет в формате HDF5.

Продемонстрировал разницу: сравнил размер файла на диске и скорость загрузки обратно в Pandas по сравнению с исходным CSV.

In [65]:
df.to_csv('my_df_to_csv', index=False)

In [66]:
df.to_hdf('my_df.h5', key='df', format='table')

In [67]:
os.path.getsize('my_df.h5') / 1024 / 1024, 'MB'

(30.594664573669434, 'MB')

In [68]:
os.path.getsize('my_df_to_csv') / 1024 / 1024, 'MB'

(34.251898765563965, 'MB')

In [69]:
%%timeit
pd.read_hdf('my_df.h5', key='df')

181 ms ± 1.89 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [70]:
%%timeit
pd.read_csv('my_df_to_csv')

334 ms ± 2.52 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [71]:
# Память до/после
print(df.memory_usage(deep=True).sum() / 1024 / 1024, 'MB — после категоризации')

58.69676971435547 MB — после категоризации


HDF5: 30.6 MB, чтение - 181 мс.
CSV: 34.3 MB, чтение - 334 мс.
HDF5 читается в 1.85 раза быстрее и занимает на 11% меньше места.
Главное преимущество: HDF5 сохраняет типы данных (datetime, category) - 
при загрузке из CSV их пришлось бы конвертировать заново.